# 01 - Data Exploration
Explore the downloaded papers, citation contexts, processed splits, and audio manifests.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_ROOT = Path().resolve().parents[1]
RAW_DIR = PROJECT_ROOT / 'src' / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'src' / 'data' / 'processed'
AUDIO_DIR = PROJECT_ROOT / 'src' / 'data' / 'audio'

## 1. Raw Papers

In [ ]:
with open(RAW_DIR / 'papers.json', 'r', encoding='utf-8') as f:
    papers = json.load(f)

papers_df = pd.DataFrame(papers)
print(f'Total papers: {len(papers_df)}')
papers_df[['title', 'year', 'citationCount']].head(10)

In [ ]:
# Year distribution
papers_df['year'].dropna().astype(int).value_counts().sort_index().plot(
    kind='bar', figsize=(12, 4), title='Papers by Year'
)
plt.tight_layout()
plt.show()

In [ ]:
# Citation count distribution
papers_df['citationCount'].dropna().clip(upper=500).hist(
    bins=50, figsize=(10, 4)
)
plt.title('Citation Count Distribution (clipped at 500)')
plt.xlabel('Citation Count')
plt.ylabel('Frequency')
plt.show()

## 2. Citation Contexts

In [ ]:
with open(RAW_DIR / 'citation_contexts.json', 'r', encoding='utf-8') as f:
    contexts = json.load(f)

ctx_df = pd.DataFrame(contexts)
print(f'Total citation contexts: {len(ctx_df)}')
ctx_df.head(3)

In [ ]:
# Context length distribution
ctx_df['ctx_len'] = ctx_df['citation_context'].str.len()
ctx_df['ctx_len'].clip(upper=600).hist(bins=50, figsize=(10, 4))
plt.title('Citation Context Length Distribution')
plt.xlabel('Characters')
plt.show()
print(ctx_df['ctx_len'].describe())

## 3. Processed Splits

In [ ]:
for split in ['train', 'val', 'test']:
    path = PROCESSED_DIR / f'{split}.json'
    if path.exists():
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f'{split:5s}: {len(data):>6,} examples')
    else:
        print(f'{split:5s}: NOT FOUND')

In [ ]:
# Inspect a few training examples
with open(PROCESSED_DIR / 'train.json', 'r', encoding='utf-8') as f:
    train_data = json.load(f)

for ex in train_data[:3]:
    print('Original :', ex['original_sentence'])
    print('Masked   :', ex['masked_sentence'])
    print('Citation :', ex['citation_string'])
    print('Source   :', ex['source_title'])
    print('-' * 80)

## 4. Citation String Frequency

In [ ]:
train_df = pd.DataFrame(train_data)
print(f'Unique citations : {train_df["citation_string"].nunique()}')
print(f'Unique sources   : {train_df["citing_paper_id"].nunique()}')
print('\nTop 20 most frequent citations:')
train_df['citation_string'].value_counts().head(20)

## 5. Audio Manifest

In [ ]:
manifest_path = AUDIO_DIR / 'train_manifest.json'
if manifest_path.exists():
    with open(manifest_path, 'r', encoding='utf-8') as f:
        manifest = json.load(f)
    print(f'Train audio samples: {len(manifest)}')
    print('\nSample entry:')
    print(json.dumps(manifest[0], indent=2))
else:
    print('Audio manifest not found. Run run_synthesis.py first.')

In [ ]:
# Listen to a sample audio
import librosa
import IPython.display as ipd

sample = manifest[0]
waveform, sr = librosa.load(sample['audio_path'], sr=16000)
print(f'Duration : {len(waveform)/sr:.2f}s')
print(f'Sentence : {sample["masked_sentence"]}')
ipd.Audio(waveform, rate=sr)